# 05 — Demonstração de telemetria HTTP + SQLite

Neste teste, um servidor HTTP local recebe dados históricos reais, atualiza um detector incremental e grava leituras e alertas em SQLite. **O Colab permite testar o HTTP internamente, mas não hospeda automaticamente o painel web para acesso público.** Para apresentar o painel interativo no navegador, execute os comandos do README no próprio computador.

In [ ]:
# Preparação: se executado a partir de um notebook aberto no Colab,
# carregue o ZIP do repositório gerado neste projeto (ou clone-o via GitHub).
from pathlib import Path
import os, sys, zipfile, subprocess
BASE = Path.cwd()
if not (BASE / 'iot_monitor').exists():
    git_repo = ""  # após publicar, opcional: "https://github.com/SEU_USUARIO/railpulse-metropt3.git"
    if git_repo.strip():
        subprocess.run(['git', 'clone', git_repo, '/content/railpulse-metropt3'], check=True)
        BASE = Path('/content/railpulse-metropt3')
    else:
        try:
            from google.colab import files
        except ImportError as e:
            raise RuntimeError('Abra o notebook na raiz do repositório ou no Colab e envie o ZIP do projeto.') from e
        print('Selecione o ZIP do repositório RailPulse. NÃO selecione o CSV original agora.')
        enviados = files.upload()
        nome = next((n for n in enviados if n.endswith('.zip')), None)
        assert nome, 'É necessário carregar o ZIP do projeto.'
        BASE = Path('/content/railpulse-metropt3')
        BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nome) as z:
            z.extractall(BASE)
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print('Raiz do projeto:', BASE)

In [ ]:
import json,threading,tempfile,urllib.request
from pathlib import Path
from http.server import ThreadingHTTPServer
from iot_monitor.server import Context,Handler
from iot_monitor.replay import samples,post_json
PORT=18765
servidor=ThreadingHTTPServer(('127.0.0.1',PORT),Handler)
_tmp=tempfile.TemporaryDirectory()
servidor.ctx=Context(json.loads((BASE/'parametros_etapa4.json').read_text()),Path(_tmp.name)/'railpulse.sqlite')
thread=threading.Thread(target=servidor.serve_forever,daemon=True);thread.start()
print('Servidor local rodando em',PORT)

In [ ]:
leituras=0; avisos=[]
for a in samples(BASE/'dados_demo'/'demo_f3.csv'):
    res=post_json(f'http://127.0.0.1:{PORT}/api/reading',a)
    leituras+=1
    avisos+=res.get('alerts',[])
print('Leituras HTTP:',leituras,'Alertas:',len(avisos))
with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/status') as r:
    status=json.load(r)
print('SQLite:',status['counts'])
assert leituras==303 and len(avisos)==3
print('✅ HTTP + SQLite + detecção incremental funcionando.')

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(avisos)[['timestamp','method','pressure_bar','elapsed_s']])
servidor.shutdown();servidor.server_close();_tmp.cleanup()

## Para ver o painel de verdade

No Windows, dentro da pasta do repositório: abra **dois terminais**. No primeiro, `python rodar_servidor.py`; no segundo, `python rodar_replay.py --demo F3 --speed 150 --reset`. Depois abra `http://127.0.0.1:8765` no navegador do mesmo computador. O endereço localhost só funciona na máquina em que o servidor está rodando.